# Previsão de caudal com os dados novos

Corre o pipeline completo do notebook 04 sobre as descargas NIVUS novas do Ameal e do Antanhol (`~/Downloads/Dados de Projeto/dados-novos-pra-testar-no-modelo`), que o modelo nunca viu, e compara com o teste habitual do notebook 04. O modelo prevê o **caudal total (m³/h)**; as componentes (esgoto, indireta, afluência direta) saem depois pela regra `split_flow`, como no notebook 04.

1. Gráficos dos dados novos, com as falhas e limitações de cada estação.
2. O pipeline do notebook 04: componentes, afinação, teste habitual, modelos finais.
3. Os modelos aplicados aos dados novos: caudal previsto, componentes, métricas do notebook 04 junto aos gráficos, curva de vazão e previsão d+5 a partir da última hora com caudal medido. No Ameal só entra o período em que o caudalímetro mediu (o sensor de velocidade falhou a partir de 6/10).
4. Comparação: dados nunca vistos contra o teste habitual do notebook 04.

In [1]:
import sys
from dataclasses import replace
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import requests
from IPython.display import display
from plotly.subplots import make_subplots

cwd = Path.cwd().resolve()
predict_dir = cwd if (cwd / "previsao.py").exists() else cwd / "predict inflow"
project_dir = predict_dir.parent
for path in (project_dir, predict_dir):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from pipeline_config import OPEN_METEO_ARCHIVE_URL, OPEN_METEO_FORECAST_URL, OPEN_METEO_TIMEZONE, STATIONS
from previsao import (
    DIAS_PASSADOS,
    HEIGHT_FEATURES,
    HORIZONTE_DIAS,
    MIN_LEITURAS_HORA,
    MODELOS,
    NOMES,
    PERFIL_CHUVA_FEATURES,
    PREVISAO_FEATURES,
    TARGET,
    backtest,
    componentes,
    errors,
    figure,
    fit_model,
    forecast,
    forecast_perfil_horario,
    load_flow,
    load_precip,
    predict,
    rain_features,
    rating_curve,
    rating_figure,
    scores,
    split_flow,
    training_table,
    tune,
)

DADOS_NOVOS = Path.home() / "Downloads" / "Dados de Projeto" / "dados-novos-pra-testar-no-modelo"
L_S_TO_M3_H = 3.6
pd.set_option("display.width", 160)
print("dados novos:", DADOS_NOVOS)

dados novos: /Users/marcosgrandesso/Downloads/Dados de Projeto/dados-novos-pra-testar-no-modelo


## 1. Os dados novos e as suas falhas

Cada estação é a pasta cujo nome começa pelo nome da estação; dentro dela, os CSV em pastas `DATA/`, como no estágio 01. A chuva é a horária do arquivo da Open-Meteo, desde 7 dias antes da primeira leitura para a janela de 7 dias do modelo ficar completa.

Uma leitura de 2 min não serve para medir caudal quando:

- **caudal = 0**: o medidor não regista escoamento; com altura normal é falha do sensor de velocidade;
- **altura fora do coletor**: altura 0 (sensor sem nível) ou altura igual ou acima do diâmetro (conduta em carga);
- **sem leitura**: intervalo entre leituras maior que 3 min.

São as mesmas regras do treino do notebook 04.

In [2]:
COLUNAS = {"caudal_l_s": "app1_q", "altura_m": "app1_h", "velocidade_m_s": "app1_v"}


def read_nivus(path: Path) -> pd.DataFrame:
    """Um export NIVUS: datetime, caudal (l/s e m³/h), altura e velocidade."""
    with path.open(encoding="utf-8-sig") as handle:
        header_row = next(i for i, line in enumerate(handle) if line.lower().startswith("date;time"))
    raw = pd.read_csv(path, sep=";", skiprows=header_row, encoding="utf-8-sig", dtype=str).dropna(subset=["Date", "Time"])
    frame = pd.DataFrame(
        {"datetime": pd.to_datetime(raw["Date"] + " " + raw["Time"], format="%d.%m.%Y %H:%M:%S", errors="coerce")}
    )
    for name, fragment in COLUNAS.items():
        column = next(c for c in raw.columns if fragment in c.lower())
        frame[name] = pd.to_numeric(raw[column].str.replace(",", ".", regex=False), errors="coerce")
    frame["caudal_m3_h"] = frame["caudal_l_s"] * L_S_TO_M3_H
    return frame


def load_new(estacao: str) -> pd.DataFrame:
    """Todas as leituras novas da estação, com o motivo de cada leitura não servir."""
    prefix = STATIONS[estacao]["folder"].lower()
    files = [
        f for folder in DADOS_NOVOS.iterdir() if folder.is_dir() and folder.name.lower().startswith(prefix)
        for f in sorted(folder.rglob("*"))
        if f.is_file() and f.parent.name.upper() == "DATA" and f.suffix.lower() == ".csv"
    ]
    if not files:
        raise FileNotFoundError(f"Nenhum CSV em DATA/ para {estacao} em {DADOS_NOVOS}")
    frame = (
        pd.concat([read_nivus(f) for f in files], ignore_index=True)
        .dropna(subset=["datetime"])
        .drop_duplicates("datetime", keep="last")
        .sort_values("datetime")
        .reset_index(drop=True)
    )
    diameter = float(STATIONS[estacao]["collector_diameter_m"])
    frame["altura_valida"] = (frame["altura_m"] > 0) & (frame["altura_m"] < diameter)
    frame["caudal_valido"] = frame["altura_valida"] & (frame["caudal_l_s"] > 0)
    frame["motivo"] = "ok"
    frame.loc[frame["caudal_l_s"] <= 0, "motivo"] = "caudal = 0"
    frame.loc[~frame["altura_valida"], "motivo"] = "altura fora do coletor"
    return frame


def fetch_rain(estacao: str, inicio: pd.Timestamp, fim: pd.Timestamp) -> pd.Series:
    """Chuva horária (mm) da Open-Meteo, de 7 dias antes de `inicio` até `fim`."""
    station = STATIONS[estacao]
    response = requests.get(
        OPEN_METEO_ARCHIVE_URL,
        params={
            "latitude": station["latitude"], "longitude": station["longitude"],
            "hourly": "precipitation", "timezone": OPEN_METEO_TIMEZONE,
            "start_date": f"{inicio - pd.Timedelta(days=7):%Y-%m-%d}", "end_date": f"{fim:%Y-%m-%d}",
        },
        timeout=30,
    )
    response.raise_for_status()
    hourly = response.json()["hourly"]
    series = pd.Series(hourly["precipitation"], index=pd.to_datetime(hourly["time"]), name="precip_mm", dtype=float)
    series.index.name = "datetime"
    return series.fillna(0.0)


novas, chuva = {}, {}
for estacao in STATIONS:
    novas[estacao] = flow = load_new(estacao)
    chuva[estacao] = fetch_rain(estacao, flow["datetime"].min(), flow["datetime"].max())
    print(
        f"{STATIONS[estacao]['folder']}: {len(flow)} leituras de {flow['datetime'].min():%d/%m %H:%M} "
        f"a {flow['datetime'].max():%d/%m %H:%M}"
    )

Ameal: 9340 leituras de 01/10 10:24 a 14/10 09:42


Antanhol: 9336 leituras de 01/10 11:10 a 14/10 10:20


### Resumo das falhas

Leituras sem caudal válido por motivo, e os períodos contínuos de pelo menos 1 hora sem caudal válido (incluindo buracos sem leituras).

In [3]:
SEM_LEITURA = pd.Timedelta(minutes=3)
FALHA_MINIMA = pd.Timedelta(hours=1)


def gaps(flow: pd.DataFrame) -> pd.DataFrame:
    """Períodos contínuos sem caudal válido: leituras inválidas seguidas ou buracos sem leituras."""
    rows = []
    run_start, motivos = None, set()
    previous = None
    for row in flow.itertuples(index=False):
        if previous is not None and row.datetime - previous > SEM_LEITURA:
            rows.append({"inicio": previous, "fim": row.datetime, "motivo": "sem leitura"})
        if not row.caudal_valido:
            if run_start is None:
                run_start, motivos = row.datetime, set()
            motivos.add(row.motivo)
        elif run_start is not None:
            rows.append({"inicio": run_start, "fim": row.datetime, "motivo": ", ".join(sorted(motivos))})
            run_start = None
        previous = row.datetime
    if run_start is not None:
        rows.append({"inicio": run_start, "fim": previous, "motivo": ", ".join(sorted(motivos))})
    table = pd.DataFrame(rows, columns=["inicio", "fim", "motivo"])
    table["duracao_h"] = ((table["fim"] - table["inicio"]).dt.total_seconds() / 3600).round(1)
    return table[table["fim"] - table["inicio"] >= FALHA_MINIMA].reset_index(drop=True)


falhas = {}
for estacao, flow in novas.items():
    falhas[estacao] = gaps(flow)
    print(f"=== {STATIONS[estacao]['folder']} ===")
    contagem = flow["motivo"].value_counts().rename("leituras").to_frame()
    contagem["%"] = (100 * contagem["leituras"] / len(flow)).round(1)
    display(contagem)
    print(f"períodos de pelo menos 1 h sem caudal válido: {len(falhas[estacao])}")
    display(falhas[estacao])

=== Ameal ===


,leituras,%
motivo,,
caudal = 0,6146,65.8
ok,3193,34.2
altura fora do coletor,1,0.0


períodos de pelo menos 1 h sem caudal válido: 2


,inicio,fim,motivo,duracao_h
0,2026-10-05 22:04:00,2026-10-06 00:26:00,caudal = 0,2.4
1,2026-10-06 01:04:00,2026-10-14 09:42:00,caudal = 0,200.6


=== Antanhol ===


,leituras,%
motivo,,
ok,9144,97.9
altura fora do coletor,181,1.9
caudal = 0,11,0.1


períodos de pelo menos 1 h sem caudal válido: 2


,inicio,fim,motivo,duracao_h
0,2026-10-06 03:42:00,2026-10-06 07:18:00,"altura fora do coletor, caudal = 0",3.6
1,2026-10-07 05:00:00,2026-10-07 07:32:00,"altura fora do coletor, caudal = 0",2.5


### Gráficos

Chuva, altura, velocidade e caudal medido de 2 min, todas as leituras sem filtro. As faixas cinzentas são os períodos de pelo menos 1 hora sem caudal válido da tabela acima; a linha tracejada na altura é o diâmetro do coletor.

In [4]:
def figure_falhas(estacao: str) -> go.Figure:
    """Chuva, altura, velocidade e caudal das leituras novas, com as falhas sombreadas."""
    flow, precip = novas[estacao], chuva[estacao]
    precip = precip[(precip.index >= flow["datetime"].min().floor("h")) & (precip.index <= flow["datetime"].max())]
    fig = make_subplots(
        rows=4, cols=1, shared_xaxes=True, row_heights=[0.15, 0.25, 0.25, 0.35], vertical_spacing=0.05,
        subplot_titles=("Chuva (mm/h)", "Altura (m)", "Velocidade (m/s)", "Caudal medido (m³/h)"),
    )
    fig.add_trace(go.Bar(x=precip.index, y=precip, name="Chuva", marker_color="gold"), row=1, col=1)
    for row, column, color in ((2, "altura_m", "steelblue"), (3, "velocidade_m_s", "darkorange"), (4, "caudal_m3_h", "navy")):
        fig.add_trace(
            go.Scatter(x=flow["datetime"], y=flow[column], mode="lines", line=dict(color=color, width=1), name=column),
            row=row, col=1,
        )
    fig.add_hline(
        y=float(STATIONS[estacao]["collector_diameter_m"]), line=dict(color="gray", dash="dash"),
        annotation_text="diâmetro do coletor", row=2, col=1,
    )
    for falha in falhas[estacao].itertuples():
        for row in (2, 3, 4):
            fig.add_vrect(x0=falha.inicio, x1=falha.fim, fillcolor="gray", opacity=0.15, line_width=0, row=row, col=1)
    fig.update_yaxes(autorange="reversed", row=1, col=1)
    fig.update_xaxes(dtick="D1", tickformat="%d/%m", row=4, col=1)
    fig.update_traces(yhoverformat=".3f")
    fig.update_layout(
        title_text=f"Dados novos - {STATIONS[estacao]['folder']} ({flow['datetime'].min():%d.%m.%Y} a {flow['datetime'].max():%d.%m.%Y})",
        height=900, template="plotly_white", hovermode="x unified", showlegend=False,
    )
    return fig


for estacao in STATIONS:
    figure_falhas(estacao).show()

## 2. O pipeline do notebook 04

O mesmo que `run_station` faz para cada estação, com os dados habituais (julho a 2 de setembro de 2026):

- **componentes**: indireta e esgoto pelas regras de `previsao.py` (`componentes`);
- **afinação e teste habitual**: `backtest` escolhe os parâmetros por validação cruzada nos dias de treino e testa nos últimos 30 % dos dias. O Antanhol fica assim. O Ameal tem um corte próprio: treino e teste são blocos de dias inteiros, sem horas do mesmo dia dos dois lados, e o teste é o sufixo final com no máximo 25 % das horas (o corte de 30 % dos dias punha metade das horas no teste, porque os últimos dias é que estão completos).
- **Ameal, modelo próprio**: só entram horas do Ameal. A chuva do modelo são as 3 horas anteriores e as 6 horas anteriores. A previsão só com altura não leva chuva.
- **modelos finais**: ajustados com todas as horas dessa estação e os parâmetros da afinação. O outubro fica de fora deste ajuste.

As métricas do teste habitual são as do notebook 04: erro médio absoluto (m³/h) e R² no caudal total e na afluência direta.

In [5]:
# Ameal: modelo próprio. O treino só vê horas do Ameal, nunca as do Antanhol.
# Teste = sufixo de dias inteiros com no máximo 25 % das horas. Com 30 % dos
# dias o teste ficava com metade das horas, porque o fim da série é que está completo.
# Chuva do modelo: soma das 3 horas anteriores e das 6 horas anteriores.
# A previsão só com altura não leva chuva.
AMEAL_FEATURES = {
    "nowcast": [*HEIGHT_FEATURES, "chuva_3h", "chuva_6h", "hour"],
    "nowcast_altura": [*HEIGHT_FEATURES],
    "previsao": ["chuva_3h", "chuva_6h", "hour"],
}


def corte_ameal(table: pd.DataFrame, max_fracao_horas: float = 0.25):
    """Treino antes de um dia, teste desse dia em diante. O teste não passa de 25 % das horas."""
    days = sorted(table["dia"].unique())
    counts = table.groupby("dia").size()
    n_test = 2
    horas = int(counts[days[-1]] + counts[days[-2]])
    limite = max_fracao_horas * len(table)
    while n_test < len(days) - 2 and horas + int(counts[days[-(n_test + 1)]]) <= limite:
        n_test += 1
        horas += int(counts[days[-n_test]])
    first = days[-n_test]
    train, test = table[table["dia"] < first], table[table["dia"] >= first]
    if train.empty or test.empty or train.index.max() >= test.index.min():
        raise RuntimeError("treino e teste do Ameal misturam datas ou ficaram vazios")
    return train, test


def backtest_ameal(table: pd.DataFrame):
    """Afinação só no bloco de treino; o teste é o bloco seguinte, com as features do Ameal."""
    train, test = corte_ameal(table)
    afinacao = {name: tune(train, features) for name, features in AMEAL_FEATURES.items()}
    estimativas = {
        name: predict(fit_model(train, features, afinacao[name].params), test, features)
        for name, features in AMEAL_FEATURES.items()
    }
    perfil = train.groupby("hour")[TARGET].mean()
    frame = pd.DataFrame({
        "medido": test[TARGET],
        **estimativas,
        "d5": test["hour"].map(perfil).fillna(float(train[TARGET].mean())),
    })
    return frame, afinacao, train


pipeline = {}
for estacao in STATIONS:
    label = STATIONS[estacao]["folder"]
    precip = load_precip(estacao)
    flow = load_flow(estacao)
    comp = componentes(flow, precip)
    # Dados novos do Antanhol: constantes do notebook 06 (outubro).
    # Janela de 1 dia seco antes. Indireta = mínimo dos mínimos das madrugadas
    # (1.188 m³/h); esgoto = média de caudal−indireta nos dias secos (15.060 m³/h).
    # A directa não usa a média 7.75 do 06: sai de split_flow com estas duas.
    if estacao == "antanhol":
        comp_novos = replace(comp, dias_antes=1, indireta_m3_h=1.188, esgoto_m3_h=15.059727)
    else:
        comp_novos = comp
    table = training_table(estacao)
    if estacao == "ameal":
        teste, afinacao, treino = backtest_ameal(table)
        features = AMEAL_FEATURES
    else:
        teste, afinacao = backtest(table)
        treino = None
        features = MODELOS
    pipeline[estacao] = {
        "comp": comp,
        "comp_novos": comp_novos,
        "table": table,
        "flow": flow,
        "afinacao": afinacao,
        "teste": teste,
        "erro": errors(teste, comp),
        "score": scores(teste, comp),
        "features": features,
        "modelos": {name: fit_model(table, cols, afinacao[name].params) for name, cols in features.items()},
        "perfil": table.groupby("hour")[TARGET].mean(),
    }
    p = pipeline[estacao]
    print(f"=== {label}: {len(table)} horas de treino, {table['dia'].nunique()} dias ===")
    print(f"componentes treino (N={comp.dias_antes}): indireta {comp.indireta_m3_h:.2f} m³/h, esgoto {comp.esgoto_m3_h:.2f} m³/h")
    if estacao == "antanhol":
        print(
            f"componentes dados novos (notebook 06, N={comp_novos.dias_antes}): "
            f"indireta {comp_novos.indireta_m3_h:.3f} m³/h, esgoto {comp_novos.esgoto_m3_h:.3f} m³/h "
            f"(directa = caudal − esgoto − indireta, sem a média de directa do 06)"
        )
    for name, a in afinacao.items():
        print(f"    {NOMES[name]}: {a.params} → erro CV {a.mae_cv:.2f} m³/h")
    if estacao == "ameal":
        print(
            f"treino {treino.index.min():%Y-%m-%d} → {treino.index.max():%Y-%m-%d} "
            f"({len(treino)} horas), só Ameal, anterior ao teste"
        )
    print(f"teste {teste.index.min():%Y-%m-%d} → {teste.index.max():%Y-%m-%d} ({len(teste)} horas)")
    print("erro médio absoluto (m³/h):")
    print(p["erro"].rename(index=NOMES).round(2).to_string())
    print("R²:")
    print(p["score"].rename(index=NOMES).round(3).to_string())
    print()

=== Ameal: 302 horas de treino, 22 dias ===
componentes treino (N=7): indireta 1.66 m³/h, esgoto 1.92 m³/h
    previsão (com altura): {'learning_rate': 0.03, 'max_depth': 3, 'max_iter': 100, 'min_samples_leaf': 10} → erro CV 1.01 m³/h
    previsão (só altura): {'learning_rate': 0.03, 'max_depth': 3, 'max_iter': 100, 'min_samples_leaf': 10} → erro CV 1.18 m³/h
    previsão (só chuva): {'learning_rate': 0.03, 'max_depth': None, 'max_iter': 300, 'min_samples_leaf': 20} → erro CV 1.35 m³/h
treino 2026-07-22 → 2026-08-30 (240 horas), só Ameal, anterior ao teste
teste 2026-08-31 → 2026-09-02 (62 horas)
erro médio absoluto (m³/h):
                               caudal_total  afluencia_direta
previsão (com altura)                  0.39              0.29
previsão (só altura)                   0.53              0.38
previsão (só chuva)                    0.62              0.51
previsão d+5 (perfil horário)          0.77              0.66
R²:
                               caudal_total  afluencia

=== Antanhol: 1006 horas de treino, 43 dias ===
componentes treino (N=7): indireta 1.26 m³/h, esgoto 22.58 m³/h
componentes dados novos (notebook 06, N=1): indireta 1.188 m³/h, esgoto 15.060 m³/h (directa = caudal − esgoto − indireta, sem a média de directa do 06)
    previsão (com altura): {'learning_rate': 0.03, 'max_depth': None, 'max_iter': 300, 'min_samples_leaf': 5} → erro CV 0.50 m³/h
    previsão (só altura): {'learning_rate': 0.03, 'max_depth': None, 'max_iter': 300, 'min_samples_leaf': 5} → erro CV 0.46 m³/h
    previsão (só chuva): {'learning_rate': 0.1, 'max_depth': 3, 'max_iter': 300, 'min_samples_leaf': 5} → erro CV 9.81 m³/h
teste 2026-08-21 → 2026-09-02 (303 horas)
erro médio absoluto (m³/h):
                               caudal_total  afluencia_direta
previsão (com altura)                  0.51              0.43
previsão (só altura)                   0.46              0.38
previsão (só chuva)                   13.90              9.45
previsão d+5 (perfil horário)     

## 3. Os modelos nos dados novos

As horas novas entram com pelo menos 15 leituras de caudal válido (caudal > 0 e altura dentro do coletor, as regras do treino). No Ameal o sensor de velocidade ficou a zero a partir de 6/10: há altura, mas o caudal medido não existe. Nas análises do Ameal só entra o período com caudal medido — a previsão não preenche esse buraco. As três previsões são com altura, só altura e só chuva.

O caudal medido e cada caudal previsto passam pela mesma `split_flow` do notebook 04. No Ameal as constantes são as do treino; no Antanhol, nos dados novos, são as do notebook 06: indireta 1.188 m³/h e esgoto 15.060 m³/h, com janela de 1 dia seco. A afluência directa não usa a média de directa do 06 — é o que sobra de `caudal − esgoto − indireta`. As métricas só usam as horas com caudal medido.

In [6]:
def hourly(flow: pd.DataFrame) -> pd.DataFrame:
    """Altura e caudal médios da hora, só com horas que têm caudal medido."""
    flow = flow.assign(hora=flow["datetime"].dt.floor("h"))
    validas = flow[flow["caudal_valido"]]
    out = validas.groupby("hora").agg(
        caudal_medido_m3_h=("caudal_m3_h", "mean"),
        n_caudal=("caudal_m3_h", "size"),
        altura_media_m=("altura_m", "mean"),
        altura_max_m=("altura_m", "max"),
    )
    out = out[out["n_caudal"] >= MIN_LEITURAS_HORA]
    out.index.name = "datetime"
    return out


def predict_new(estacao: str) -> pd.DataFrame:
    """Caudal medido e as três previsões (m³/h), hora a hora, com os nomes de colunas do notebook 04."""
    p = pipeline[estacao]
    table = hourly(novas[estacao]).join(rain_features(chuva[estacao]), how="inner")
    table["hour"] = table.index.hour
    table["weekday"] = table.index.dayofweek
    return pd.DataFrame(
        {
            "medido": table["caudal_medido_m3_h"],
            **{name: predict(p["modelos"][name], table, cols) for name, cols in p["features"].items()},
            "d5": table["hour"].map(p["perfil"]).fillna(float(p["table"][TARGET].mean())),
        }
    ), table


novos = {}
for estacao in STATIONS:
    p = pipeline[estacao]
    caudais, table = predict_new(estacao)
    medidas = caudais.dropna(subset=["medido"])
    novos[estacao] = {
        "caudais": caudais,
        "table": table,
        "erro": errors(medidas, p["comp_novos"]),
        "score": scores(medidas, p["comp_novos"]),
        "n_medidas": len(medidas),
    }
    print(
        f"{STATIONS[estacao]['folder']}: {len(caudais)} horas com caudal medido "
        f"({caudais.index.min():%d/%m %H:%M} → {caudais.index.max():%d/%m %H:%M})"
    )

Ameal: 108 horas com caudal medido (01/10 10:00 → 06/10 00:00)
Antanhol: 305 horas com caudal medido (01/10 11:00 → 14/10 09:00)


### Previsão d+5 a partir da última hora com caudal medido

O d+5 é o caudal médio de cada hora do relógio, calculado em tudo o que já foi medido até à última hora com caudal. Sem altura futura, este perfil erra menos do que a previsão só com chuva. No Ameal a âncora é a última hora válida (cerca de 6/10), não o fim do ficheiro. A chuva do gráfico continua a vir da Open-Meteo; o caudal previsto não depende dela.

In [7]:
def fetch_forecast_rain(estacao: str, inicio: pd.Timestamp) -> pd.Series:
    """Chuva horária (mm) da Open-Meteo, de 7 dias antes de `inicio` até d+5."""
    station = STATIONS[estacao]
    response = requests.get(
        OPEN_METEO_FORECAST_URL,
        params={
            "latitude": station["latitude"], "longitude": station["longitude"],
            "hourly": "precipitation", "timezone": OPEN_METEO_TIMEZONE,
            "start_date": f"{inicio - pd.Timedelta(days=DIAS_PASSADOS):%Y-%m-%d}",
            "end_date": f"{inicio + pd.Timedelta(days=HORIZONTE_DIAS):%Y-%m-%d}",
        },
        timeout=30,
    )
    response.raise_for_status()
    hourly_rain = response.json()["hourly"]
    series = pd.Series(hourly_rain["precipitation"], index=pd.to_datetime(hourly_rain["time"]), name="precip_mm", dtype=float)
    series.index.name = "datetime"
    return series.fillna(0.0)


for estacao in STATIONS:
    p, n = pipeline[estacao], novos[estacao]
    inicio = n["caudais"].index.max() + pd.Timedelta(hours=1)
    n["inicio_d5"] = inicio
    chuva_d5 = fetch_forecast_rain(estacao, inicio)
    medido = n["caudais"]["medido"].dropna()
    serie = pd.concat([
        p["table"][[TARGET, "hour"]],
        pd.DataFrame({TARGET: medido, "hour": medido.index.hour}),
    ])
    n["futuro"] = forecast_perfil_horario(serie, chuva_d5, p["comp_novos"], inicio)
    print(f"{STATIONS[estacao]['folder']}: d+5 de {inicio:%d/%m %H:%M} a {n['futuro'].index.max():%d/%m %H:%M}")

Ameal: d+5 de 06/10 01:00 a 11/10 00:00


Antanhol: d+5 de 14/10 10:00 a 19/10 09:00


### Gráficos com as métricas

Para cada estação: as métricas do notebook 04 nos dados novos, o gráfico do notebook 04 (chuva, caudal total e afluência direta; dados novos à esquerda, d+5 à direita) com as métricas escritas por baixo, e a curva de vazão das leituras novas comparada com a do treino.

In [8]:
def metrics_text(erro: pd.DataFrame, score: pd.DataFrame, titulo: str) -> str:
    """As métricas do notebook 04 em linhas de texto para uma anotação plotly."""
    linhas = [f"<b>{titulo}</b>"]
    for nome, tabela, coluna, fmt in (
        ("R² caudal total", score, "caudal_total", ".3f"),
        ("Erro médio abs. caudal total (m³/h)", erro, "caudal_total", ".2f"),
        ("R² afluência direta", score, "afluencia_direta", ".3f"),
        ("Erro médio abs. afluência direta (m³/h)", erro, "afluencia_direta", ".2f"),
    ):
        linhas.append(f"{nome}: " + " · ".join(f"{NOMES[s]} {tabela.loc[s, coluna]:{fmt}}" for s in NOMES))
    return "<br>".join(linhas)


def add_metrics(fig: go.Figure, text: str) -> None:
    """Escreve as métricas por baixo dos eixos, no canto inferior esquerdo."""
    fig.add_annotation(
        text=text, xref="paper", yref="paper", x=0, y=-0.06, xanchor="left", yanchor="top",
        showarrow=False, align="left", bgcolor="white", bordercolor="gray", borderwidth=1, font=dict(size=12),
    )
    fig.update_layout(margin=dict(b=170), height=(fig.layout.height or 600) + 130)


for estacao in STATIONS:
    label = STATIONS[estacao]["folder"]
    p, n = pipeline[estacao], novos[estacao]
    c = n["caudais"]
    titulo = f"Dados novos {c.index.min():%d/%m} → {c.index.max():%d/%m} ({n['n_medidas']} h com caudal medido)"
    print(f"=== {label} — métricas nos dados novos ({n['n_medidas']} horas com caudal medido) ===")
    print("erro médio absoluto (m³/h):")
    print(n["erro"].rename(index=NOMES).round(2).to_string())
    print("R²:")
    print(n["score"].rename(index=NOMES).round(3).to_string())

    chuva_fig = pd.concat([
        chuva[estacao][(chuva[estacao].index >= c.index.min()) & (chuva[estacao].index < n["inicio_d5"])],
        n["futuro"]["precip_mm"],
    ])
    fig = figure(label, c, n["futuro"], chuva_fig, p["comp_novos"], None, n["erro"], n["score"])
    fig.update_layout(title_text=f"Previsão de afluências - {label} (dados novos à esquerda, d+5 à direita)")
    add_metrics(fig, metrics_text(n["erro"], n["score"], titulo))
    fig.show()

    validas = novas[estacao][novas[estacao]["caudal_valido"]]
    horas = n["table"].assign(caudal_total_m3_h=c["medido"]).dropna(subset=["caudal_total_m3_h"])
    a_treino, b_treino = rating_curve(p["flow"])
    a_novo, b_novo = rating_curve(validas)
    curva = rating_figure(f"{label} (dados novos)", validas, horas, float(STATIONS[estacao]["collector_diameter_m"]))
    alturas = pd.Series(sorted(validas["altura_m"]))
    curva.add_trace(go.Scatter(
        x=alturas, y=a_treino * alturas**b_treino, name=f"Ajuste do treino: Q = {a_treino:.1f} · h^{b_treino:.2f}",
        mode="lines", line=dict(color="black", width=2, dash="dash"),
    ))
    add_metrics(curva, metrics_text(n["erro"], n["score"], titulo))
    print(f"curva de vazão: treino Q = {a_treino:.1f} · h^{b_treino:.2f} | dados novos Q = {a_novo:.1f} · h^{b_novo:.2f}")
    curva.show()

=== Ameal — métricas nos dados novos (108 horas com caudal medido) ===
erro médio absoluto (m³/h):
                               caudal_total  afluencia_direta
previsão (com altura)                  0.75              0.64
previsão (só altura)                   0.89              0.76
previsão (só chuva)                    1.19              1.02
previsão d+5 (perfil horário)          1.11              0.92
R²:
                               caudal_total  afluencia_direta
previsão (com altura)                 0.708             0.675
previsão (só altura)                  0.672             0.636
previsão (só chuva)                  -0.052            -0.174
previsão d+5 (perfil horário)         0.022            -0.061


curva de vazão: treino Q = 3635.7 · h^2.71 | dados novos Q = 34385.7 · h^3.53


=== Antanhol — métricas nos dados novos (305 horas com caudal medido) ===
erro médio absoluto (m³/h):
                               caudal_total  afluencia_direta
previsão (com altura)                  0.14              0.05
previsão (só altura)                   0.13              0.05
previsão (só chuva)                   10.48              5.83
previsão d+5 (perfil horário)         10.36              5.54
R²:
                               caudal_total  afluencia_direta
previsão (com altura)                 1.000             0.999
previsão (só altura)                  1.000             0.999
previsão (só chuva)                  -1.322            -2.751
previsão d+5 (perfil horário)        -0.922            -1.464


curva de vazão: treino Q = 30360.8 · h^2.08 | dados novos Q = 39558.4 · h^2.15
